# employee data analysis

## 1. Setup and Imports

In [ ]:
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
CUR_DIR = Path.cwd().resolve()


### 1.1 load dataset

In [ ]:
df_raw = pd.read_csv(CUR_DIR / 'dataset/employee_2023.csv', sep=',', index_col=None)

## 2. Initial Data Exploration - Getting to Know Our Data

Before diving into cleaning, we need to understand what we're working with.


In [ ]:
print("=" * 80)
print(" INITIAL DATA EXPLORATION")
print("=" * 80)

print(f"   Features: {df_raw.dtypes}")
print(f"\n First look at the data:")
display(df_raw.head())

print(f"\n Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\n Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Data types and completeness
print("\n Data Types and Completeness:")
dtype_summary = pd.DataFrame({
    'Data Type': df_raw.dtypes,
    'Non-Null Count': df_raw.notnull().sum(),
    'Null Count': df_raw.isnull().sum(),
    'Null %': (df_raw.isnull().sum() / len(df_raw) * 100).round(1),
    'Unique Values': df_raw.nunique()
})

# Numerical columns overview
print("\n Quick Statistical Summary (Numeric Columns):")
display(df_raw.describe().round(2))

# Categorical columns overview
print("\n Categorical Columns Overview:")
cat_cols = df_raw.select_dtypes(include=['object']).columns
for col in cat_cols:
    if col != 'name':
        print(f"\n{col}:")
        print(df_raw[col].value_counts().to_string()) # todo: hire_date too long (shwo first couple rows) 


## 3. Data Quality Assessment - Systematic Problem Finding
Now we audit every column against rules.
### Good data rules:
1. COMPLETENESS - All required values are present
2. VALIDITY - Values follow business rules
3. CONSISTENCY - Values make sense together
4. UNIQUENESS - No unwanted duplicates
5. ACCURACY - Values reflect reality (hard to verify)

In [ ]:

class DataQualityAssessor:
    """
    data quality assessment.
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.rules = self._define_rules()
        self.quality_report = {}
        
    def _define_rules(self) -> dict:
        """Define business rules that data must satisfy."""
        return {
            "Age must be positive (18-70)": 
                (self.df["age"] > 0) & (self.df["age"] <= 70),
            "Salary must be reasonable ($20k-$500k)": 
                (self.df["salary"] > 20000) & (self.df["salary"] < 500000),
            "Performance score must be 0-100": 
                (self.df["performance_score"] >= 0) & (self.df["performance_score"] <= 100),
            "Projects completed must be non-negative": 
                self.df["projects_completed"] >= 0,
            "Overtime hours must be non-negative": 
                self.df["overtime_hours"] >= 0,
            "Tenure must be non-negative": 
                self.df["tenure_years"] >= 0,
            "Team size must be positive": 
                self.df["team_size"] > 0,
            "Hire date cannot be in the future": 
                pd.to_datetime(self.df['hire_date']).dt.date <= datetime.now().date(),
            "Satisfaction must be a valid category": 
                self.df["satisfaction"].isin(['Very Low', 'Low', 'Medium', 'High', 'Very High'])
        }
    
    def assess(self) -> dict:
        """Run quality assessment."""
        
        # Check completeness
        completeness = pd.DataFrame({
            'Column': self.df.columns,
            'Missing': self.df.isnull().sum(),
            'Complete %': ((1 - self.df.isnull().sum() / len(self.df)) * 100).round(1)
        }).sort_values('Missing', ascending=False)
        
        # Check validity (business rules)
        violations = {}
        for rule, condition in self.rules.items():
            violations[rule] = (~condition.fillna(True)).sum()
        
        # Check uniqueness
        duplicates = self.df.duplicated().sum()
        id_duplicates = self.df['employee_id'].duplicated().sum()
        
        # quality score
        total_checks = len(self.df) * len(self.rules)
        total_violations = sum(violations.values())
        quality_score = (1 - total_violations / total_checks) * 100
        
        self.quality_report = {
            'completeness': completeness,
            'violations': pd.Series(violations),
            'duplicate_rows': duplicates,
            'duplicate_ids': id_duplicates,
            'quality_score': quality_score
        }
        
        return self.quality_report
    
    def print_report(self):
        """Print a quality report."""
        if not self.quality_report:
            self.assess()
        
        r = self.quality_report
        
        print("=" * 80)
        print(" DATA QUALITY ASSESSMENT REPORT")
        print("=" * 80)
        
        print(f"\n Overall Quality Score: {r['quality_score']:.1f}%")
        
        print(f"\n Completeness Issues:")
        missing = r['completeness'][r['completeness']['Missing'] > 0]
        if len(missing) > 0:
            display(missing)
        else:
            print(" No missing values found")
        
        print(f"\n Business Rule Violations:")
        violations = r['violations'][r['violations'] > 0]
        if len(violations) > 0:
            for rule, count in violations.items():
                pct = (count / len(self.df)) * 100
                print(f"   • {rule}: {count} violations ({pct:.1f}%)")
        else:
            print(" No rule violations found")
        
        print(f"\n Uniqueness:")
        print(f"   - Duplicate rows: {r['duplicate_rows']}")
        print(f"   - Duplicate employee IDs: {r['duplicate_ids']}")

assessor = DataQualityAssessor(df_raw)
assessor.assess()
assessor.print_report()